# Dense Neural Netowrks

## Notation

* $f_k$ is the output of layer $k$ before activation
* $h_{k+1}$ is $f_k$ after activation
* $x = h_0$
* $Ω_k$ and $β_k$ are the the weights matrix and the bias vector respectively
* $Ω_k$ is of the shape $(n(h_{k+1}), n(h_k))$


## Forward Pass

$$
f_k = Ω_kh_k + β_k \\
h_{k+1} = a[f_k]
$$

### Neural Networks as a Computational Graph
* Each operation in a neural network can be written as a network of operations performed on the input vector. This forms a **Directed Acyclic Graph**
* We can Toposort DAGs to find the dependencies of each operation. This gives us a representation of **neural networks as a composition of functions**.


## Backpropagation

### Loss Function
* $L(\hat{y}, y)$ is defined as the error between the predicted and expected output
* MSE for regression, BCE for binary classification, CCE for multi-class classification

### Algorithm
* Let the neural network be $$x → f_0 → h_1 → f_1 → h_2 → f_2 → h_3 → l$$ (2 hidden layers, $h_3$ is output)
* $\frac{∂l}{∂f_i}$ is what we need to find (we can find $\frac{∂l}{∂Ω_i}$ and $\frac{∂l}{∂β_i}$ from this term using chain rule)

$$
\frac{∂l}{∂h_3} = \text{derivative of loss function w.r.t }h_3 \\
\frac{∂l}{∂f_2} = \frac{∂h_3}{∂f_2}\frac{∂l}{∂h_3} \\
\frac{∂l}{∂f_1} = \frac{∂h_2}{∂f_1}\frac{∂f_2}{∂h_2}\frac{∂l}{∂f_2} \\
\frac{∂l}{∂f_0} = \frac{∂h_1}{∂f_0}\frac{∂f_1}{∂h_1}\frac{∂l}{∂f_1} \\
$$

* After we have all the $\frac{∂l}{∂f_i}$ terms, we can do:
$$
f_k = Ω_kh_k + β_k \\
\frac{∂f_k}{∂Ω_k} = \frac{∂l}{∂f_k}\frac{∂f_k}{∂Ω_k}  = \frac{∂l}{∂f_k} h_k^T \\
\frac{∂f_k}{∂β_k } = \frac{∂l}{∂f_k}\frac{∂f_k}{∂β_k }  = \frac{∂l}{∂f_k}
$$

* These terms are used to update weights in different optimization algorithms

* **Note:** $\frac{∂l}{∂f_k} = \frac{∂l}{∂h_{k+1}} ⊙ a'(f_k)$

# **Optimizers**


1. Batch Gradient Descent
2. Stochastic Gradient Descent and Mini-Batch Gradient Descent
3. Momentum Gradient Descent
4. Nesterov Accelerated Gradient Descent
5. AdaGrad
6. RMSProp
7. Adam
8. Coordinate Descent

### **Batch Gradient Descent**


* An algorithm used to optimize a given model on the basis of its cost function.
* It takes the partial derivative of the loss function with respect to its current weights and uses that to minimize the cost.

#### **Working**
* Let $J(w, b)$ be the cost function, where $w$ is the weights vector and $b$ is the bias term
* Let $α$ be the learning rate
  * It helps control the pace at which the algorithm converges. Too fast, and the algorithm might overshoot and oscillate about its minima forever.

##### **Algorithm**
$$w_j := w_j - α\frac{∂J(w, b)}{∂w_j}\\
b := b - α\frac{∂J(w, b)}{∂b}
$$
* where $w_j$ is the $j^{th}$ weight in the weight vector
* The algorithm is repeated till convergence to the local/global minima

### **Stochastic Gradient Descent and Mini-Batch Gradient Descent**


#### **Stochastic Gradient Descent (SGD)**
* Uses the same algorithm as the regular gradient descent, but updates weights based on loss instead of cost.
* **Loss vs Cost:**
  * Loss is calculated for each sample
  * cost is the average loss over all samples
* This means that in SGD, based on the loss in each sample, the weights get updated

##### **Pros**
* This method adds a bit of randomness (the loss is calculated for different samples, so for some samples, loss might be high, for others low). This helps models **escape local minima and reach the global minima**.
* Since the weights are updated per sample, **convergence is faster**.

#### **Mini-Batch Gradient Descent**
* Same as SGD, but instead of using every sample independently, the model uses the cost over smaller subsets of data

##### **Pros**
* Control randomness and offers a mixed approach that is both stable and efficient

### **Momentum Gradient Descent**


* The algorithm builds up a certain "momentum" in the general direction that the model was going in.
  * This is similar to a rock rolling down a slope. Even if the rock enocounters a valley, it can overcome it due to the momentum it has built up as it rolled down. Therefore, **unless a deep enough minima is found, the model can escape the local minima**

#### **Algorithm**
$$
v_{t+1} = β v_t + (1-β)∇J(w_t)\\
w_{t+1} = w_t - αv_{t+1}
$$
* where $t$ represents the number of iterations the algorithm has run.
* [Why is the formula like that](https://towardsdatascience.com/gradient-descent-with-momentum-59420f626c8f/)

### **Nesterov Accelerated Gradient Descent**

* Modification of Momentum Gradient Descent

#### **Algorithm**
$$
v_{t+1} = β v_t + α∇J(w_t - βv_t)\\
w_{t+1} = w_t - v_{t+1}
$$

#### **Pros**
* Faster convergence due to the predictive nature of the momentum term.
* More stable optimization by reducing oscillations and overshooting.
* Better performance in handling plateaus and complex, high-dimensional loss surfaces.
* Improved generalization with faster training and fewer iterations.
* Efficiency in deep learning, where optimization landscapes are highly non-convex.

### **AdaGrad**


* AdaGrad is a modification of Gradient descent that adaptively modifies the learning rate.
* It scales the learning rate by the inverse of the RMS (root mean squared) of the previous gradients (When the algorithm is in a plateau, it moves faster since the RMS is smaller than when the algorithm is in a steep slope, where it moves very slowly so as to not miss the minima)
* It doesn't have a momentum aspect, so it will get stuck at local minima, if any.

#### **Algorithm**
$$
G_{t+1} = \sum^{t}_{i=1}g_i^2\\
w_{t+1} = w_t - \frac{1}{\sqrt{G_{t+1} + ϵ}}g_{t}
$$
* $g_t$ is $∇J(w)$ at the $t^{th}$ iteration
* $ϵ$ is a very small value like $10^{-8}$ to avoid divison by 0

### **RMSProp**


1. Learning Rate Adaptation:
  * RMSProp adapts the learning rate for each parameter, based on the recent magnitudes of the gradients. This means it adjusts the step size (learning rate) according to how large or small the gradients are.
  * If the gradient is large, it reduces the learning rate, and if the gradient is small, it increases the learning rate. (minimizes chance of osciallations and traverses plateaus faster)
2. Running Average of Squared Gradients:
  * Instead of using just the gradient value (like in regular gradient descent), RMSProp keeps a running average of the squared gradients. This is done using an exponentially decaying average.

  * The idea is that if the gradient for a parameter has been large for some time, the learning rate for that parameter will decrease, and vice versa.
  * RMSProp lacks any momentum aspect
#### **Working**
$$
v_t = βv_{t-1} + (1-β)(∇J(w_{t-1}))^2\\
w_t = w_{t-1} - \frac{α}{\sqrt{v_t}+ ϵ}∇J(w_{t-1})
$$
  * $ϵ$ is a very small value like $10^{-8}$ to prevent division by 0

### **Adam**



* **Adam** (short for **Adaptive Moment Estimation**) is an optimization algorithm that combines the **benefits of momentum** and **adaptive learning rates** (like RMSProp). It's widely used in deep learning because it's **efficient**, **robust**, and **works well out of the box**.



#### **Working Rules**

Adam maintains:

1. **First moment (mean) of gradients** → like **momentum**
2. **Second moment (uncentered variance)** of gradients → like **RMSProp**

It uses these two "memories" to adapt the learning rate **per parameter**, while also smoothing updates over time.

**Definitions**
* $g_t$: gradient at time step $t$
* $m_t$: first moment (mean of gradients)
* $v_t$: second moment (mean of squared gradients)
* $\beta_1$: decay rate for first moment (usually 0.9)
* $\beta_2$: decay rate for second moment (usually 0.999)
* $\epsilon$: small constant to avoid division by zero (usually $10^{-8}$)
* $\eta$: learning rate

#### **Algorithm**

1. **Compute gradient**:

   $$
   g_t = \nabla_\theta J(\theta_t)
   $$

2. **Update biased first moment estimate (momentum-like)**:

   $$
   m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t
   $$

3. **Update biased second moment estimate (RMS-like)**:

   $$
   v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2
   $$

4. **Bias correction** (because $m_t$, $v_t$ are initialized at 0):

   $$
   \hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}
   $$

5. **Update parameters**:

   $$
   \theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
   $$


#### Advantages

* **Combines benefits of RMSProp and momentum**.
* **Works well with sparse gradients** (like in NLP).
* **Requires little hyperparameter tuning** (defaults work well).
* **Adaptive per-parameter updates** — scales well for complex models.


#### Disadvantages

* **Can converge to suboptimal solutions** in some cases (especially compared to SGD in some convex or very smooth problems).
* **Can overfit** — sometimes doesn’t generalize as well as SGD with momentum.
* Sensitive to **learning rate schedule** in certain problems.


### **Coordinate Descent**

* Does gradient descent by fixing all the other weights but one to arbitrary values.
* Memory efficient and scalable for high-dimensional data